# Data Preprocessing

## import libraies

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

In [2]:
df = pd.read_csv('..\Data\Dopamine_D2_receptor_02_bioactivity_data_raw.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\HP\AppData\Local\Temp\ipykernel_11764\53568396.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('..\Data\Dopamine_D2_receptor_02_bioactivity_data_raw.csv')


In [7]:
df.isna().sum()

molecule_chembl_id    0
canonical_smiles      0
standard_value        0
dtype: int64

## Clean Molecules from saltes

In [4]:
smiles = []
for i in df['canonical_smiles'].tolist():
    mol = Chem.MolFromSmiles(str(i))
    
    # If RDKit can't read the molecule, just keep the original text
    if mol is None:
        smiles.append(i)
        continue
        
    # Split into fragments and find the one with the most heavy atoms
    frags = Chem.GetMolFrags(mol, asMols=True)
    largest_frag = max(frags, key=Chem.rdchem.Mol.GetNumHeavyAtoms)
    
    smiles.append(Chem.MolToSmiles(largest_frag))

df['canonical_smiles'] = smiles

In [6]:

df[df['canonical_smiles'] == None]

,molecule_chembl_id,canonical_smiles,standard_value


## **Calculate Lipinski descriptors**
Christopher Lipinski, a scientist at Pfizer, came up with a set of rule-of-thumb for evaluating the **druglikeness** of compounds. Such druglikeness is based on the Absorption, Distribution, Metabolism and Excretion (ADME) that is also known as the pharmacokinetic profile. Lipinski analyzed all orally active FDA-approved drugs in the formulation of what is to be known as the **Rule-of-Five** or **Lipinski's Rule**.

The Lipinski's Rule stated the following:
* Molecular weight < 500 Dalton
* Octanol-water partition coefficient (LogP) < 5
* Hydrogen bond donors < 5
* Hydrogen bond acceptors < 10 

In [8]:
# Inspired by: https://codeocean.com/explore/capsules?query=tag:data-curation

def lipinski(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem) 
        moldata.append(mol)
       
    baseData= np.arange(1,1)
    i=0  
    for mol in moldata:        
       
        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)
           
        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])   
    
        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1      
    
    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]   
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)
    
    return descriptors

In [9]:
df_lipinski = lipinski(df.canonical_smiles)
df_lipinski

,MW,LogP,NumHDonors,NumHAcceptors
0,342.446,3.3700,0.0,4.0
1,360.461,3.4744,0.0,5.0
2,365.427,4.3490,0.0,3.0
3,360.461,3.4744,0.0,5.0
4,336.464,3.5273,0.0,5.0
...,...,...,...,...
1565,388.895,3.5755,0.0,4.0
1566,372.465,2.7866,0.0,5.0
1567,392.503,3.0707,1.0,4.0
1568,393.487,3.4034,1.0,4.0


In [10]:
df_combined = pd.concat([df,df_lipinski], axis=1)

### **Convert IC50 to pIC50**
To allow **IC50** data to be more uniformly distributed, we will convert **IC50** to the negative logarithmic scale which is essentially **-log10(IC50)**.

This custom function pIC50() will accept a DataFrame as input and will:
* Take the IC50 values from the ``standard_value`` column and converts it from nM to M by multiplying the value by 10$^{-9}$
* Take the molar value and apply -log10
* Delete the ``standard_value`` column and create a new ``pIC50`` column

In [15]:
def norm_value(input):
    norm = []

    for i in input['standard_value']:
        if i > 100000000:
          i = 100000000
        norm.append(i)

    input['standard_value_norm'] = norm
    x = input.drop('standard_value', axis=1)
        
    return x

In [18]:
def pIC50(input):
    pIC50 = []

    for i in input['standard_value_norm']:
        molar = i*(10**-9) # Converts nM to M
        pIC50.append(-np.log10(molar))

    input['pIC50'] = pIC50
    x = input.drop('standard_value_norm', axis=1)
        
    return x

In [16]:
df_norm = norm_value(df_combined)

In [17]:
df_norm.standard_value_norm.describe()

count    1.570000e+03
mean     1.038817e+04
std      2.747752e+04
min      3.000000e-10
25%      4.461750e+01
50%      1.000000e+03
75%      1.000000e+04
max      5.800000e+05
Name: standard_value_norm, dtype: float64

In [19]:
df_final = pIC50(df_norm)

In [20]:
df_final.pIC50.describe()

count    1570.000000
mean        6.284053
std         1.562643
min         3.236572
25%         5.000000
50%         6.000000
75%         7.350495
max        18.522879
Name: pIC50, dtype: float64

### As we see in our data the range before the log transformation was between 3*10e-10 to 5800000 and after transformation it became from 3.23 to 18.52 which will increase the model performance

In [21]:
df_final.to_csv('..\Data\Dopamine_D2_receptor_03_bioactivity_data_raw.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\HP\AppData\Local\Temp\ipykernel_11764\3750017268.py:1: SyntaxWarning: invalid escape sequence '\D'
  df_final.to_csv('..\Data\Dopamine_D2_receptor_03_bioactivity_data_raw.csv')
